In [ ]:

import os
import numpy as np
import pandas as pd

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
cm_path = os.path.join(
    project_dir, "evaluation", "confusion_matrix.csv"
)

cm_df = pd.read_csv(cm_path, index_col=0)

cm = cm_df.to_numpy()
labels = cm_df.index.astype(str).to_numpy()

# Remove correct predictions from consideration
errors = cm.copy()
np.fill_diagonal(errors, 0)

# Find the most frequent incorrect class pairs
rows, cols = np.where(errors > 0)

pairs = pd.DataFrame({
    "actual_gloss": labels[rows],
    "predicted_gloss": labels[cols],
    "misclassified_samples": errors[rows, cols]
})

pairs = pairs.sort_values(
    "misclassified_samples", ascending=False
).head(15)

print("TOP 15 CONFUSION PAIRS")
print(pairs.to_string(index=False))

# Overall correct vs incorrect predictions
total = int(cm.sum())
correct = int(np.trace(cm))
incorrect = total - correct

print("\nPREDICTION SUMMARY")
print("Total:", total)
print("Correct:", correct)
print("Incorrect:", incorrect)
print("Error rate:", round(incorrect / total * 100, 2), "%")


In [ ]:

import numpy as np

print("FEATURE AVAILABILITY CHECK\n")

for name in ["X_train", "X_val", "X_test", "y_train", "y_val", "y_test"]:
    if name in globals():
        value = globals()[name]
        print(f"{name}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"{name}: NOT AVAILABLE")

if "X_train" in globals():
    print("\nFeature dimensions:")
    print("Hand features: first 126 values")
    print("Pose features: last 132 values")

    print("\nTraining feature finite check:")
    print("All finite:", bool(np.isfinite(X_train).all()))

    print("\nNonzero values by feature group:")
    print("First 126:", int(np.count_nonzero(X_train[:, :, :126])))
    print("Last 132:", int(np.count_nonzero(X_train[:, :, 126:])))


In [ ]:

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, f1_score

tf.keras.utils.set_random_seed(42)

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
ablation_dir = os.path.join(project_dir, "ablation_experiment")
os.makedirs(ablation_dir, exist_ok=True)

feature_sets = {
    "hand_only": (0, 126),
    "pose_only": (126, 258),
    "combined": (0, 258)
}

results = []

def build_model(input_features):
    network = models.Sequential([
        layers.Input(shape=(8, input_features)),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(100, activation="softmax")
    ])

    network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return network

for name, (start, end) in feature_sets.items():
    print(f"\n{'=' * 45}")
    print("EXPERIMENT:", name)

    Xtr = X_train[:, :, start:end]
    Xv = X_val[:, :, start:end]
    Xte = X_test[:, :, start:end]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    ablation_model = build_model(end - start)

    callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )

    history = ablation_model.fit(
        Xtr, y_train,
        validation_data=(Xv, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[callback],
        verbose=1
    )

    probabilities = ablation_model.predict(Xte, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(
        y_test, predictions,
        average="macro", zero_division=0
    )

    best_val_accuracy = max(history.history["val_accuracy"])

    results.append({
        "experiment": name,
        "feature_count": end - start,
        "epochs_trained": len(history.history["loss"]),
        "best_validation_accuracy": float(best_val_accuracy),
        "test_accuracy": float(accuracy),
        "test_macro_f1": float(macro_f1)
    })

    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Test macro-F1: {macro_f1:.4f}")

    del ablation_model

results_df = pd.DataFrame(results)
results_path = os.path.join(ablation_dir, "ablation_results.csv")
results_df.to_csv(results_path, index=False)

print("\nFINAL ABLATION COMPARISON")
print(results_df.round(4).to_string(index=False))
print("\nSaved results:", results_path)


In [ ]:

import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
output_dir = os.path.join(project_dir, "hand_only_model")
os.makedirs(output_dir, exist_ok=True)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

hand_model = models.Sequential([
    layers.Input(shape=(8, 126)),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(100, activation="softmax")
])

hand_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(
    output_dir, "best_hand_only_model.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history_hand = hand_model.fit(
    X_train[:, :, :126],
    y_train,
    validation_data=(X_val[:, :, :126], y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

history_path = os.path.join(output_dir, "training_history.csv")
import pandas as pd
pd.DataFrame(history_hand.history).to_csv(
    history_path, index=False
)

config = {
    "experiment": "hand-only landmark LSTM",
    "input_shape": [8, 126],
    "output_classes": 100,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 32,
    "maximum_epochs": 20,
    "selection_metric": "validation loss",
    "test_set_used_during_training": False
}

with open(
    os.path.join(output_dir, "model_config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print("\nSaved best model:", checkpoint_path)
print("Saved training history:", history_path)
print("Saved model configuration.")
print("Best validation accuracy:",
      round(max(history_hand.history["val_accuracy"]), 4))


In [ ]:

import os
import json
import pandas as pd
import numpy as np

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"

# 1. Verify saved model and experiment artifacts
required_files = [
    "asl_100class_landmark_lstm.keras",
    "asl_100class_label_mapping.json",
    "experiment_results.json",
    "training_history.csv",
    "recovered_feature_manifest.csv",
    "experiment_diagnostics.json",
    "evaluation/classification_report.csv",
    "evaluation/confusion_matrix.csv",
    "evaluation/test_predictions.csv",
    "ablation_experiment/ablation_results.csv",
    "hand_only_model/best_hand_only_model.keras",
    "hand_only_model/training_history.csv",
    "hand_only_model/model_config.json"
]

print("FINAL ARTIFACT CHECK\n")
missing = []

for filename in required_files:
    path = os.path.join(project_dir, filename)
    if os.path.isfile(path):
        print(f"[OK] {filename}")
    else:
        print(f"[MISSING] {filename}")
        missing.append(filename)

# 2. Save a consolidated experiment summary
summary = {
    "project": "Signer-Independent ASL Sign Recognition",
    "task": "100-class landmark-based gloss classification",
    "recovered_records": 3150,
    "expected_records": 3218,
    "missing_records": 68,
    "baseline_test_accuracy": 0.1312,
    "baseline_macro_f1": 0.1088,
    "hand_only_ablation_test_accuracy": 0.1423,
    "hand_only_ablation_macro_f1": 0.1091,
    "pose_only_test_accuracy": 0.0372,
    "pose_only_macro_f1": 0.0067,
    "combined_ablation_test_accuracy": 0.0828,
    "combined_ablation_macro_f1": 0.0474,
    "hand_only_checkpoint_saved": os.path.isfile(
        os.path.join(
            project_dir,
            "hand_only_model/best_hand_only_model.keras"
        )
    ),
    "missing_artifacts": missing
}

summary_path = os.path.join(
    project_dir, "final_experiment_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSummary saved:", summary_path)
print("Missing artifacts:", len(missing))
print("Verification complete.")


In [ ]:

import os
import zipfile

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
zip_path = os.path.join(
    project_dir, "ASL_Research_Artifacts.zip"
)

include_files = [
    "asl_100class_landmark_lstm.keras",
    "asl_100class_label_mapping.json",
    "experiment_results.json",
    "training_history.csv",
    "recovered_feature_manifest.csv",
    "experiment_diagnostics.json",
    "final_experiment_summary.json",
    "evaluation/classification_report.csv",
    "evaluation/confusion_matrix.csv",
    "evaluation/test_predictions.csv",
    "evaluation/evaluation_summary.json",
    "ablation_experiment/ablation_results.csv",
    "hand_only_model/best_hand_only_model.keras",
    "hand_only_model/training_history.csv",
    "hand_only_model/model_config.json"
]

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for relative_path in include_files:
        full_path = os.path.join(project_dir, relative_path)
        if os.path.isfile(full_path):
            archive.write(full_path, arcname=relative_path)
            print("[ADDED]", relative_path)
        else:
            print("[SKIPPED]", relative_path)

print("\nZIP created:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))
print("Project artifacts packaged.")


In [ ]:
from pathlib import Path
import json
import os

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

print("PROJECT DIRECTORY:", PROJECT_DIR)
print("=" * 70)

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{size_mb:8.2f} MB | {path.relative_to(PROJECT_DIR)}")

print("\n" + "=" * 70)
print("DATASET / FEATURE FILE SUMMARY")

for path in PROJECT_DIR.rglob("*"):
    if path.is_file() and any(
        word in path.name.lower()
        for word in ["manifest", "feature", "dataset", "label", "participant"]
    ):
        print(f"{path.name}: {path.stat().st_size:,} bytes")

print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")

summary_path = PROJECT_DIR / "final_experiment_summary.json"

if summary_path.exists():
    with open(summary_path, "r") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Summary file not found.")

print("\nInspection complete.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from collections import Counter

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
BATCH_DIR = PROJECT_DIR / "batches"
MANIFEST_PATH = PROJECT_DIR / "recovered_feature_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)
print("MANIFEST SHAPE:", manifest.shape)
print("MANIFEST COLUMNS:", manifest.columns.tolist())
print("\nMISSING VALUES:\n", manifest.isna().sum())
print("\nSPLIT COUNTS:\n", manifest["split"].value_counts(dropna=False))
print("\nUNIQUE GLOSSES:", manifest["Gloss"].nunique())
print("UNIQUE PARTICIPANTS:", manifest["Participant ID"].nunique())

print("\n" + "=" * 60)
print("BATCH VALIDATION")

batch_files = sorted(BATCH_DIR.glob("batch_*.npz"))
print("BATCH FILES:", len(batch_files))

shape_counts = Counter()
array_keys = Counter()
total_records = 0
invalid_records = 0
label_samples = []
batch_errors = []

for path in batch_files:
    try:
        with np.load(path, allow_pickle=False) as data:
            keys = list(data.files)
            for key in keys:
                array_keys[key] += 1

            print(f"\n{path.name}")
            for key in keys:
                arr = data[key]
                print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

                if arr.ndim > 0:
                    shape_counts[(key, tuple(arr.shape[1:]))] += arr.shape[0]

                if arr.dtype.kind in "fiu":
                    if not np.isfinite(arr).all():
                        invalid_records += 1
                        print("  WARNING: NaN or Inf values detected")

                if arr.ndim > 0:
                    total_records = max(total_records, arr.shape[0])

            for key in keys:
                if any(term in key.lower() for term in ["label", "gloss", "participant", "video", "id"]):
                    values = data[key]
                    label_samples.append((path.name, key, values.reshape(-1)[:5].tolist()))

    except Exception as e:
        batch_errors.append((path.name, str(e)))
        print("  ERROR:", e)

print("\n" + "=" * 60)
print("ARRAY KEYS:", dict(array_keys))
print("OBSERVED ARRAY SHAPES:")
for item, count in shape_counts.items():
    print(item, "records:", count)

print("\nSAMPLE METADATA ARRAYS:")
for sample in label_samples[:20]:
    print(sample)

print("\nBATCH ERRORS:", batch_errors)
print("NUMERICAL WARNINGS:", invalid_records)

print("\n" + "=" * 60)
print("MANIFEST SAMPLE")
print(manifest.head(10).to_string(index=False))

print("\nINSPECTION COMPLETE. No files were modified.")

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

metadata_rows = []
feature_arrays = []

for path in sorted((PROJECT_DIR / "batches").glob("batch_*.npz")):
    with np.load(path, allow_pickle=False) as data:
        feature_arrays.append(data["features"])
        for item in data["metadata"]:
            try:
                metadata_rows.append(json.loads(str(item)))
            except Exception:
                metadata_rows.append({"_raw_metadata": str(item)})

X = np.concatenate(feature_arrays, axis=0)
meta = pd.DataFrame(metadata_rows)
manifest = pd.read_csv(PROJECT_DIR / "recovered_feature_manifest.csv")

print("FEATURE ARRAY:", X.shape)
print("METADATA ROWS:", len(meta))
print("\nFIRST RAW METADATA:")
print(metadata_rows[0])

print("\nMETADATA COLUMNS:")
print(meta.columns.tolist())

print("\nMETADATA SAMPLE:")
print(meta.head(5).to_string(index=False))

print("\nMANIFEST COLUMNS:")
print(manifest.columns.tolist())

print("\nDUPLICATE METADATA ROWS:", meta.duplicated().sum())
print("DUPLICATE MANIFEST VIDEOS:", manifest["Video file"].duplicated().sum())

print("\nFEATURE STATISTICS:")
print("Min:", float(X.min()))
print("Max:", float(X.max()))
print("Mean:", float(X.mean()))
print("Std:", float(X.std()))

print("\nMISSING FEATURE VALUES PER SAMPLE:")
missing = np.isnan(X).any(axis=(1, 2)).sum()
print(missing)

print("\nINSPECTION COMPLETE")

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

X_parts, metadata_rows = [], []

for path in sorted((PROJECT_DIR / "batches").glob("batch_*.npz")):
    with np.load(path, allow_pickle=False) as data:
        X_parts.append(data["features"])
        metadata_rows.extend(json.loads(str(m)) for m in data["metadata"])

X = np.concatenate(X_parts).astype(np.float32)
df = pd.DataFrame(metadata_rows)

# Confirm manifest and batch metadata agree in the same row order
manifest = pd.read_csv(PROJECT_DIR / "recovered_feature_manifest.csv")
print("Metadata equals manifest:", df.equals(manifest))
print("Features:", X.shape)

# Class balance
print("\nEXAMPLES PER SPLIT:")
print(df["split"].value_counts())

print("\nCLASS DISTRIBUTION:")
print(df.groupby(["split", "Gloss"]).size().groupby("split").describe())

print("\nLEAST REPRESENTED CLASSES:")
print(df["Gloss"].value_counts().sort_values().head(15))

print("\nMOST REPRESENTED CLASSES:")
print(df["Gloss"].value_counts().sort_values(ascending=False).head(15))

# Signer overlap across splits
signers = {
    split: set(df.loc[df["split"] == split, "Participant ID"])
    for split in ["train", "val", "test"]
}

print("\nSIGNER COUNTS:")
for split, ids in signers.items():
    print(split, len(ids))

print("\nSIGNER OVERLAP:")
print("train/val:", signers["train"] & signers["val"])
print("train/test:", signers["train"] & signers["test"])
print("val/test:", signers["val"] & signers["test"])

# Check whether every class occurs in each split
classes = set(df["Gloss"])
for split in ["train", "val", "test"]:
    present = set(df.loc[df["split"] == split, "Gloss"])
    print(f"{split}: {len(present)}/{len(classes)} classes")

# Save verified reconstructed feature archive
out_path = PROJECT_DIR / "verified_reconstructed_features.npz"
np.savez_compressed(
    out_path,
    X=X,
    gloss=df["Gloss"].to_numpy(dtype=str),
    participant=df["Participant ID"].to_numpy(dtype=str),
    video=df["Video file"].to_numpy(dtype=str),
    split=df["split"].to_numpy(dtype=str)
)

df.to_csv(PROJECT_DIR / "verified_reconstructed_manifest.csv", index=False)

print("\nSaved:", out_path)
print("Manifest saved.")

In [ ]:
import numpy as np
import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

data = np.load(
    PROJECT_DIR / "verified_reconstructed_features.npz",
    allow_pickle=False
)

X = data["X"]
gloss = data["gloss"]
participants = data["participant"]
splits = data["split"]

print("Feature shape:", X.shape)
print("Feature dtype:", X.dtype)
print("All finite:", np.isfinite(X).all())

print("\nPer-frame feature statistics:")
print("Mean:", round(float(X.mean()), 4))
print("Std:", round(float(X.std()), 4))
print("Min:", round(float(X.min()), 4))
print("Max:", round(float(X.max()), 4))

print("\nFeature dimensions:")
print("Frames per sequence:", X.shape[1])
print("Features per frame:", X.shape[2])
print("Expected structure: 258 features per frame")

print("\nSplit counts:")
for split in ["train", "val", "test"]:
    mask = splits == split
    print(
        split,
        "| samples:", int(mask.sum()),
        "| signers:", len(set(participants[mask])),
        "| classes:", len(set(gloss[mask]))
    )

print("\nSaved archive verified.")

In [ ]:

import numpy as np
import pandas as pd
import json
import tensorflow as tf

from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

data = np.load(
    PROJECT_DIR / "verified_reconstructed_features.npz",
    allow_pickle=False
)

X = data["X"].astype(np.float32)
gloss = data["gloss"].astype(str)
splits = data["split"].astype(str)

train_mask = splits == "train"
val_mask = splits == "val"
test_mask = splits == "test"

encoder = LabelEncoder()
encoder.fit(gloss[train_mask])

y = encoder.transform(gloss)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Fit normalization using training data only.
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

def normalize_sequences(features):
    shape = features.shape
    return scaler.transform(
        features.reshape(-1, shape[-1])
    ).reshape(shape).astype(np.float32)

X_train = normalize_sequences(X_train)
X_val = normalize_sequences(X_val)
X_test = normalize_sequences(X_test)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(encoder.classes_)),
    y=y_train
)

class_weights = {
    i: float(weight) for i, weight in enumerate(weights)
}

np.save(
    PROJECT_DIR / "label_classes.npy",
    encoder.classes_
)

np.savez_compressed(
    PROJECT_DIR / "train_only_scaler.npz",
    mean=scaler.mean_,
    scale=scaler.scale_
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Classes:", len(encoder.classes_))
print("All normalized values finite:",
      np.isfinite(X_train).all()
      and np.isfinite(X_val).all()
      and np.isfinite(X_test).all())
print("Class weights calculated.")


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import accuracy_score, f1_score

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
MODEL_DIR = PROJECT_DIR / "model_comparison"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(42)

NUM_CLASSES = len(encoder.classes_)
INPUT_SHAPE = X_train.shape[1:]

def build_mlp():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

def build_lstm():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

def build_cnn_lstm():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.TimeDistributed(
            layers.Dense(128, activation="relu")
        ),
        layers.Conv1D(128, 3, padding="same", activation="relu"),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

builders = {
    "MLP": build_mlp,
    "LSTM": build_lstm,
    "CNN_LSTM": build_cnn_lstm
}

results = []

for name, builder in builders.items():
    print(f"\n{'='*15} Training {name} {'='*15}")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = builder()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    checkpoint_path = MODEL_DIR / f"best_{name.lower()}.keras"

    training_callbacks = [
        callbacks.ModelCheckpoint(
            str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            verbose=0
        ),
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        )
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=80,
        batch_size=32,
        class_weight=class_weights,
        callbacks=training_callbacks,
        verbose=1
    )

    best_model = tf.keras.models.load_model(checkpoint_path)
    val_probs = best_model.predict(X_val, verbose=0)
    val_pred = np.argmax(val_probs, axis=1)

    val_accuracy = accuracy_score(y_val, val_pred)
    val_macro_f1 = f1_score(
        y_val, val_pred, average="macro", zero_division=0
    )

    results.append({
        "model": name,
        "best_val_accuracy": val_accuracy,
        "best_val_macro_f1": val_macro_f1,
        "epochs_trained": len(history.history["loss"]),
        "checkpoint": str(checkpoint_path)
    })

    pd.DataFrame(history.history).to_csv(
        MODEL_DIR / f"history_{name.lower()}.csv",
        index=False
    )

    print(f"{name} validation accuracy: {val_accuracy:.4f}")
    print(f"{name} validation macro-F1: {val_macro_f1:.4f}")

results_df = pd.DataFrame(results).sort_values(
    ["best_val_macro_f1", "best_val_accuracy"],
    ascending=False
)

results_df.to_csv(MODEL_DIR / "validation_comparison.csv", index=False)

print("\nFINAL VALIDATION COMPARISON")
print(results_df.to_string(index=False))
print("\nSaved results to:", MODEL_DIR)


In [ ]:

import numpy as np
import pandas as pd
import json
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score
)

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
MODEL_DIR = PROJECT_DIR / "model_comparison"
EVAL_DIR = PROJECT_DIR / "final_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(
    MODEL_DIR / "best_lstm.keras"
)

# Evaluate the untouched test set once.
test_probabilities = model.predict(X_test, verbose=1)
test_predictions = np.argmax(test_probabilities, axis=1)

test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test, test_predictions,
    average="macro", zero_division=0
)
test_weighted_f1 = f1_score(
    y_test, test_predictions,
    average="weighted", zero_division=0
)
test_top5 = top_k_accuracy_score(
    y_test,
    test_probabilities,
    k=5,
    labels=np.arange(NUM_CLASSES)
)

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=encoder.classes_,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(EVAL_DIR / "classification_report.csv")

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(NUM_CLASSES)
)
pd.DataFrame(
    cm,
    index=encoder.classes_,
    columns=encoder.classes_
).to_csv(EVAL_DIR / "confusion_matrix.csv")

predictions_df = pd.DataFrame({
    "true_gloss": encoder.inverse_transform(y_test),
    "predicted_gloss": encoder.inverse_transform(test_predictions),
    "correct": y_test == test_predictions,
    "top5_glosses": [
        ", ".join(encoder.classes_[idx] for idx in row)
        for row in np.argsort(test_probabilities, axis=1)[:, -5:][:, ::-1]
    ]
})
predictions_df.to_csv(
    EVAL_DIR / "test_predictions.csv",
    index=False
)

summary = {
    "model": "LSTM",
    "number_of_classes": int(NUM_CLASSES),
    "test_samples": int(len(y_test)),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "test_top5_accuracy": float(test_top5),
    "validation_accuracy": 0.326996,
    "validation_macro_f1": 0.311823
}

with open(EVAL_DIR / "evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nFINAL TEST RESULTS")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\nBEST-PERFORMING CLASSES")
print(
    report_df.loc[encoder.classes_, ["precision", "recall", "f1-score"]]
    .sort_values("f1-score", ascending=False)
    .head(10)
)

print("\nWEAKEST-PERFORMING CLASSES")
print(
    report_df.loc[encoder.classes_, ["precision", "recall", "f1-score"]]
    .sort_values("f1-score")
    .head(10)
)

print("\nEvaluation files saved in:", EVAL_DIR)


In [ ]:

import json
import numpy as np
import tensorflow as tf
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
MODEL_DIR = PROJECT_DIR / "final_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(
    PROJECT_DIR / "model_comparison" / "best_lstm.keras"
)

model.save(MODEL_DIR / "asl_signer_independent_lstm.keras")
np.save(MODEL_DIR / "label_classes.npy", encoder.classes_)

np.savez_compressed(
    MODEL_DIR / "preprocessing_scaler.npz",
    mean=scaler.mean_,
    scale=scaler.scale_
)

with open(MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump({
        "model_type": "LSTM",
        "num_classes": int(len(encoder.classes_)),
        "sequence_length": int(X_train.shape[1]),
        "features_per_frame": int(X_train.shape[2]),
        "test_accuracy": 0.2558,
        "test_macro_f1": 0.2443,
        "test_top5_accuracy": 0.6056
    }, f, indent=2)

print("Final model and preprocessing assets saved.")
print(MODEL_DIR)


In [ ]:

import numpy as np
import tensorflow as tf
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
MODEL_DIR = PROJECT_DIR / "final_model"

model = tf.keras.models.load_model(
    MODEL_DIR / "asl_signer_independent_lstm.keras"
)

classes = np.load(MODEL_DIR / "label_classes.npy", allow_pickle=False)

scaler_data = np.load(
    MODEL_DIR / "preprocessing_scaler.npz",
    allow_pickle=False
)

scaler_mean = scaler_data["mean"]
scaler_scale = scaler_data["scale"]

def predict_sign(sequence, top_k=5):
    sequence = np.asarray(sequence, dtype=np.float32)

    if sequence.shape != (8, 258):
        raise ValueError("Input must have shape (8, 258).")

    if not np.isfinite(sequence).all():
        raise ValueError("Input contains NaN or infinite values.")

    normalized = (sequence - scaler_mean) / scaler_scale
    probabilities = model.predict(normalized[np.newaxis, ...], verbose=0)[0]
    indices = np.argsort(probabilities)[::-1][:top_k]

    return [
        {
            "gloss": str(classes[index]),
            "probability": float(probabilities[index])
        }
        for index in indices
    ]

sample_predictions = predict_sign(X_test[0])
print("Actual gloss:", encoder.inverse_transform([y_test[0]])[0])
print("Predictions:")
for item in sample_predictions:
    print(item)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
EVAL_DIR = PROJECT_DIR / "final_evaluation"

cm_df = pd.read_csv(
    EVAL_DIR / "confusion_matrix.csv",
    index_col=0
)

cm = cm_df.to_numpy()

plt.figure(figsize=(18, 16))
plt.imshow(cm, interpolation="nearest", aspect="auto")
plt.title("LSTM Test Confusion Matrix")
plt.xlabel("Predicted Gloss")
plt.ylabel("True Gloss")
plt.colorbar()
plt.tight_layout()
plt.savefig(EVAL_DIR / "confusion_matrix.png", dpi=200)
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
MODEL_DIR = PROJECT_DIR / "model_comparison"

for name in ["mlp", "lstm", "cnn_lstm"]:
    history_path = MODEL_DIR / f"history_{name}.csv"
    history = pd.read_csv(history_path)

    plt.figure(figsize=(8, 5))
    plt.plot(history["accuracy"], label="Training")
    plt.plot(history["val_accuracy"], label="Validation")
    plt.title(f"{name.upper()} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(MODEL_DIR / f"{name}_accuracy.png", dpi=160)
    plt.show()


In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

readme = """# Signer-Independent ASL Gloss Recognition

## Project Overview

This project classifies isolated American Sign Language signs into 100 gloss classes using landmark sequences and an LSTM neural network.

## Dataset

The project uses extracted landmark features from the ASL Citizen dataset. The current recovered feature set contains 3,150 sequences across 100 classes.

## Input Representation

Each sample contains 8 frames, with 258 numerical features per frame.

## Data Splitting

The dataset is split into training, validation, and test subsets. Participant IDs were checked for overlap, and no participant overlap was found between these subsets.

## Model Selection

MLP, LSTM, and CNN-LSTM architectures were compared using validation performance. The LSTM was selected based on validation macro-F1.

## Final Test Results

- Test accuracy: 25.58%
- Test macro-F1: 24.43%
- Test weighted-F1: 24.00%
- Test top-5 accuracy: 60.56%

## Limitations

The model does not perform continuous sign-language translation, sentence generation, or full language understanding. It classifies isolated signs from pre-extracted landmarks. Some classes have zero F1-score, and performance is not yet reliable enough for practical communication.

## Reproducibility

The final model requires the matching class-label array and training-fitted normalization parameters. Raw-video inference additionally requires a compatible landmark extraction pipeline.

## Dataset License

Follow the ASL Citizen dataset's original license and research-use restrictions. Do not redistribute original videos or restricted data.
"""

(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("README saved:", PROJECT_DIR / "README.md")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(PROJECT_DIR)} | {path.stat().st_size / (1024 * 1024):.2f} MB")


In [ ]:

import zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ASL_Citizen_Project")
ZIP_PATH = PROJECT_DIR / "ASL_Final_Model_and_Evaluation.zip"

include_dirs = [
    PROJECT_DIR / "final_model",
    PROJECT_DIR / "final_evaluation",
    PROJECT_DIR / "model_comparison"
]

include_files = [
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "verified_reconstructed_manifest.csv",
    PROJECT_DIR / "final_experiment_summary.json"
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in include_files:
        if file_path.exists():
            archive.write(file_path, file_path.relative_to(PROJECT_DIR))

    for folder in include_dirs:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(PROJECT_DIR))

print("Package created:", ZIP_PATH)
print("Size:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2), "MB")


In [ ]:

import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/ASL_Citizen_Project/ASL_Final_Model_and_Evaluation.zip")

with zipfile.ZipFile(zip_path, "r") as archive:
    files = archive.namelist()
    bad_file = archive.testzip()

print("ZIP exists:", zip_path.exists())
print("ZIP integrity:", "PASS" if bad_file is None else f"FAIL: {bad_file}")
print("Total files:", len(files))

for file in files:
    print(file)


In [ ]:

from google.colab import files

files.download("/content/drive/MyDrive/ASL_Citizen_Project/ASL_Final_Model_and_Evaluation.zip")


In [ ]:
import os
import json
import numpy as np

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"

print("PROJECT FILES")
for root, dirs, files in os.walk(project_dir):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for file in files:
        if file.endswith((".py", ".ipynb", ".json", ".npz", ".npy")):
            print(os.path.relpath(os.path.join(root, file), project_dir))

print("\nFEATURE SHAPE")
data = np.load(os.path.join(project_dir, "verified_reconstructed_features.npz"))
print("Available arrays:", data.files)

for key in data.files:
    arr = data[key]
    print(key, "shape:", arr.shape, "dtype:", arr.dtype)

print("\nNOTEBOOKS CONTAINING FEATURE EXTRACTION CODE")
for root, dirs, files in os.walk(project_dir):
    for file in files:
        if file.endswith(".ipynb"):
            path = os.path.join(root, file)
            with open(path, "r", encoding="utf-8") as f:
                notebook = json.load(f)

            matches = []
            for i, cell in enumerate(notebook.get("cells", [])):
                source = "".join(cell.get("source", []))
                if any(term in source.lower() for term in [
                    "mediapipe", "pose_landmarker", "hand_landmarker",
                    "258", "landmark", "feature extraction"
                ]):
                    matches.append((i, source[:1800]))

            if matches:
                print("\nNOTEBOOK:", path)
                for i, source in matches[:12]:
                    print(f"\nCELL {i}\n{source}\n")

In [ ]:
import os
import json
import numpy as np

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"

files_to_check = [
    "experiment_diagnostics.json",
    "experiment_results.json",
    "final_experiment_summary.json",
    "evaluation/evaluation_summary.json",
    "final_evaluation/evaluation_summary.json",
    "hand_only_model/model_config.json",
    "final_model/model_metadata.json"
]

for rel_path in files_to_check:
    path = os.path.join(project_dir, rel_path)
    if os.path.exists(path):
        print(f"\n{'=' * 15} {rel_path} {'=' * 15}")
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2)[:6000])

batch_dir = os.path.join(project_dir, "batches")
batch_files = sorted(
    f for f in os.listdir(batch_dir) if f.endswith(".npz")
) if os.path.isdir(batch_dir) else []

print("\nNumber of batch files:", len(batch_files))

if batch_files:
    with np.load(os.path.join(batch_dir, batch_files[0]), allow_pickle=False) as batch:
        print("Batch arrays:", batch.files)
        for key in batch.files:
            print(key, batch[key].shape, batch[key].dtype)